# Docking poses and reference frame correspondence

Public DockingMT/MolSysMT/MolSysViewer integration control using native Vina and
181L. Preparation is explicitly provisional; this is not a validated affinity
benchmark. Assertions inspect Python molecular geometry, regions and player state.
They do not certify browser rendering. See [the contract](viewer_reference.md) for
source profiles, legacy limitations and the provider issue handoff.


In [1]:
from pathlib import Path
import hashlib
import inspect
import json
import platform
import importlib.metadata as metadata
import numpy as np
import pyunitwizard as puw
import molsysmt as msm
import molsysviewer as msv
import dockingmt as dmt
from molsysviewer_dockingmt import build_docking_reference_system, ensure_runtime

path = Path(msm.systems['T4 lysozyme L99A']['181l.pdb'])
profile = {
    'python': platform.python_version(),
    'vina': metadata.version('vina'),
    'molsysmt_origin': msm.__file__,
    'molsysviewer_origin': msv.__file__,
    'explicit_pairing': 'structure_pairing' in inspect.signature(msv.MolSysView.load).parameters,
    '181l_sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
}
print(json.dumps(profile, indent=2))


{
  "python": "3.14.7",
  "vina": "1.2.7",
  "molsysmt_origin": "/home/diego/repos@uibcdf/dockingmt/.pytest_cache/codex-source-archives/dockingmt-template-provider-c19a47ada0c2279029abfa296cf915560610ad9a/molsysmt/__init__.py",
  "molsysviewer_origin": "/home/diego/repos@uibcdf/dockingmt/.pytest_cache/codex-source-archives/dockingmt-template-viewer-ec4c71e574d798b7c8675b7e7e983da878ce9889/molsysviewer/__init__.py",
  "explicit_pairing": false,
  "181l_sha256": "77018feaaa65bb22dea47c784e8c059b0ccc09cd6dc7442b79cce83f3170985f"
}


In [2]:
molecular_system = msm.convert(str(path), to_form='molsysmt.MolSys')
reference = msm.extract(molecular_system, selection="group_name=='BNZ'")
domain = dmt.BoxRegion.from_selection(
    molecular_system, selection="group_name=='BNZ'",
    padding=puw.quantity(8, 'angstrom'),
)
problem = dmt.DockingProblem(
    receptor=molecular_system, partner=molecular_system,
    receptor_selection="molecule_type=='protein'",
    partner_selection="group_name=='BNZ'", search_domain=domain,
)
result = dmt.dock(problem, dmt.VinaProtocol(
    seed=42, cpu=1, exhaustiveness=1, n_poses=3,
    energy_range=puw.quantity(10, 'kcal/mol'),
    allow_provisional_preparation=True,
))
assert len(result.poses) > 1
assert result.top_pose.metadata['pose_atom_order'] == 'verified_pdbqt_order'
assert result.provenance['preparation']['partner']['assessment'] == 'provisional'
print({'poses': len(result.poses), 'preparation': 'provisional', 'seed': 42})


{'poses': 3, 'preparation': 'provisional', 'seed': 42}


In [3]:
prepared_reference = build_docking_reference_system(reference, len(result.poses))
assert msm.get(prepared_reference, n_structures=True) == len(result.poses)
expected = np.repeat(
    puw.get_value(msm.get(reference, coordinates=True), to_unit='angstrom'),
    len(result.poses), axis=0,
)
viewer = dmt.view(result, reference=reference)
reference_region = viewer.regions.get('reference_ligand')
np.testing.assert_allclose(
    puw.get_value(reference_region.get(element='atom', coordinates=True), to_unit='angstrom'),
    expected,
)
assert viewer.player.n_structures == len(result.poses)
atom_count = viewer.whole.get(n_atoms=True)
for rank in range(len(result.poses), 0, -1):
    msv_runtime = ensure_runtime(viewer)
    from molsysviewer_dockingmt import set_active_pose
    set_active_pose(viewer, rank)
    assert viewer.player.index == rank - 1
    assert msv_runtime.active_pose_rank == rank

# Reload the result: replacement prevents duplicate molecular components.
dmt.view(result, reference=prepared_reference, view=viewer)
assert viewer.whole.get(n_atoms=True) == atom_count
assert viewer.player.index == 0
assert viewer.regions.contains('reference_ligand')
print({'pose_frames': viewer.player.n_structures, 'atoms_after_reload': atom_count,
       'reference_frames': len(expected), 'active_pose_rank': ensure_runtime(viewer).active_pose_rank})


{'pose_frames': 3, 'atoms_after_reload': 1301, 'reference_frames': 3, 'active_pose_rank': 1}


In [4]:
bad_reference = msm.extract(reference, structure_indices=[0] * (len(result.poses) + 1))
previous = ensure_runtime(viewer).result
try:
    dmt.view(result, reference=bad_reference, view=viewer)
except ValueError as error:
    print({'rejected_reference': str(error)})
else:
    raise AssertionError('A mismatched reference was accepted')
assert ensure_runtime(viewer).result is previous
assert viewer.whole.get(n_atoms=True) == atom_count
assert viewer.regions.contains('reference_ligand')
assert msm.get(reference, n_structures=True) == 1
print('Frame-count rejection occurs before scene replacement; source unchanged.')


ERROR: The reference must have 1 or 3 structures; got 4.


{'rejected_reference': 'The reference must have 1 or 3 structures; got 4.'}
Frame-count rejection occurs before scene replacement; source unchanged.
